# 07 – Native-speaker evaluation: retrieval → d1 gate (Kaggle)

Frozen configuration: nothing is tuned here. Retriever = multilingual-e5-large; gate = d1-3B with `gate_questions_v2.json`;
temperatures/thresholds are applied later by `scripts/analyze_native.py` from `configs/gate_frozen.json`.

Settings: Accelerator = GPU (T4), Internet = On, Input = a dataset containing
`native_eval_v1.csv`, `AISIS_KnowledgeBase_v1.xlsx`, `gate_questions_v2.json`.
Output: `results_native.csv` (download link in the last cell) → `results/native/results_native_v1.csv`.

Two passage conditions per instance: **retrieved** (top-1 of the retriever; the real end-to-end setting) and **gold**
(only for the 27 base queries that have a gold passage; separates retrieval errors from gate errors).

In [ ]:
!pip install -q -U sentence-transformers "transformers>=5.14" accelerate

In [ ]:
import glob, json, time, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
f = lambda n: glob.glob(f"/kaggle/input/**/{n}", recursive=True)[0]
N = pd.read_csv(f("native_eval_v1.csv")).fillna("")
kb = pd.read_excel(f("AISIS_KnowledgeBase_v1.xlsx"), sheet_name="KB")
QUESTIONS = json.load(open(f("gate_questions_v2.json"), encoding="utf-8"))["questions"]
TOOLS = {k: v for k, v in QUESTIONS["tool"]["criteria"].items() if k != "none"}
pids, texts = kb.passage_id.tolist(), kb.text.tolist()
print(len(N), "instances;", N.version.value_counts().to_dict(), "|", len(pids), "passages")  # expect 209 instances, 136 passages

In [ ]:
# --- retrieval (multilingual-e5-large, "query: " / "passage: " prefixes) ---
enc = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
P = enc.encode(["passage: " + t for t in texts], normalize_embeddings=True, batch_size=32)
Q = enc.encode(["query: " + q for q in N["query"]], normalize_embeddings=True, batch_size=32)
S = Q @ P.T
order = np.argsort(-S, axis=1)
N["top1_id"] = [pids[o[0]] for o in order]
N["top1_sim"] = [float(S[i, o[0]]) for i, o in enumerate(order)]
N["top3_ids"] = [json.dumps([pids[j] for j in o[:3]]) for o in order]
gi = {p: i for i, p in enumerate(pids)}
N["gold_rank"] = [int(np.where(order[i] == gi[g])[0][0]) + 1 if g else 0 for i, g in enumerate(N.gold_passage_id)]
has = N[N.gold_passage_id != ""]
print("R@1 by version (gold-passage queries):"); print(has.groupby("version").gold_rank.apply(lambda s: (s == 1).mean()).round(3).to_dict())
del enc; torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModel
def load(dtype): return AutoModel.from_pretrained("LiquidAI/d1-3B", trust_remote_code=True, dtype=dtype).to("cuda")
model = load(torch.float16)
t = model.system_one("test", {"q": {"type": "choice", "instructions": "pick", "criteria": {"a": "first", "b": "second"}}})
p = t["answers"]["q"]["probabilities"]; vals = list(p.values()) if isinstance(p, dict) else list(p)
if any(v != v for v in vals):
    del model; torch.cuda.empty_cache(); model = load(torch.float32); print("fell back to fp32")
ptext = dict(zip(pids, texts))
jobs = []
for r in N.to_dict("records"):
    jobs.append(("retrieved", r, ptext[r["top1_id"]]))
    if r["gold_passage_id"]:
        jobs.append(("gold", r, ptext[r["gold_passage_id"]]))
print(len(jobs), "gate calls (5 questions each)")

In [ ]:
rows, t0 = [], time.time()
for cond, r, passage in jobs:
    state = {"user_query": r["query"], "retrieved_passage": passage, "available_tools": TOOLS}
    ans = model.system_one(state, QUESTIONS)["answers"]
    row = {k: r[k] for k in ["id","type","topic","version","native_status","query","gold_action","gold_tool","gold_args",
                             "gold_passage_id","top1_id","top1_sim","top3_ids","gold_rank"]}
    row["passage_condition"] = cond
    for q in QUESTIONS:
        a = ans[q]
        row[f"pred_{q}"] = a.get("choice"); row[f"conf_{q}"] = a.get("confidence")
        row[f"probs_{q}"] = json.dumps(a.get("probabilities"), ensure_ascii=False, default=str)
    rows.append(row)
el = time.time() - t0
print(f"{el:.1f}s, {1000*el/len(jobs):.0f} ms per instance; peak GPU GB {torch.cuda.max_memory_allocated()/1e9:.1f}")
res = pd.DataFrame(rows); res.to_csv("/kaggle/working/results_native.csv", index=False)
print(res.groupby(["passage_condition","version"]).size().unstack(0))

In [ ]:
import base64
from IPython.display import HTML
d = base64.b64encode(open("/kaggle/working/results_native.csv","rb").read()).decode()
HTML(f'<a download="results_native.csv" href="data:text/csv;base64,{d}">Download results_native.csv</a>')